# V2 — Fine-tuning de Qwen3-1.7B con LoRA

LoRA fine-tuning de Qwen3-1.7B sobre un subset de 10.000 muestras de CNN/DailyMail.
Solo entrenamos los adaptadores de rango bajo (r=16) sobre las proyecciones de
atención — la base queda congelada. Objetivo: superar el baseline zero-shot
(ROUGE-1 = 22.56).

**Configuración:**
- LoRA r=16, alpha=32, dropout=0.05
- Target modules: q_proj, k_proj, v_proj, o_proj
- bf16 + gradient checkpointing (para caber cómodamente en 12 GB VRAM)
- Batch efectivo = 16 (per_device=1 × grad_accum=16)
- 1 epoch, learning rate 2e-4 (más alto que full FT — típico para LoRA)
- `max_input_length=1024` durante training (vs 1536 en inferencia) para reducir
  presión de VRAM. Esto eleva la truncación del 5.6% al ~15-20%, aún mucho mejor
  que el 73% de Flan-T5.
- Sin eval durante training (beam search sería prohibitivamente caro).
  La evaluación con ROUGE se hace una sola vez al final.


In [4]:
# Setup
import sys
import os
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

import torch
from src.data.loader import load_config, load_cnn_dailymail
from src.data.preprocessing import build_causal_preprocess_fn, tokenize_dataset
from src.models.loader import load_model
from src.training.trainer import apply_lora, build_causal_trainer

print(f"CUDA: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0)}")
cfg = load_config("../config/config.yaml")


CUDA: True | NVIDIA GeForce RTX 5070


## 1. Training

In [ ]:
# Load dataset and model, then wrap with LoRA.
# 1 epoch × 10k is enough for LoRA to converge on this task.
cfg["dataset"]["train_subset"] = 10000
cfg["training"]["num_train_epochs"] = 1

# Reduce training context from 1536 to 1024 tokens to ease VRAM pressure.
# Inference will still use the full 1536 from the config.
TRAIN_MAX_INPUT = 1024

dataset = load_cnn_dailymail(cfg)
loaded = load_model(cfg["models"]["qwen"])

preprocess_fn = build_causal_preprocess_fn(
    tokenizer=loaded.tokenizer,
    max_input_length=TRAIN_MAX_INPUT,
    max_target_length=cfg["dataset"]["max_target_length"],
)
tokenized = tokenize_dataset(dataset, preprocess_fn)

loaded = apply_lora(loaded, cfg)  # prints trainable params
print(tokenized)


In [ ]:
# Build trainer. Gradient checkpointing is enabled inside build_causal_trainer.
trainer = build_causal_trainer(
    loaded=loaded,
    tokenized_dataset=tokenized,
    cfg=cfg,
    output_subdir="v2_qwen_lora",
    per_device_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
)


In [ ]:
# Train! Expect ~2-3h on RTX 5070 with gradient checkpointing.
# No eval during training — we run a single ROUGE evaluation after.
train_result = trainer.train()
trainer.save_model()
trainer.save_metrics("train", train_result.metrics)
print(train_result.metrics)


## 2. Post-training evaluation

El modelo se recarga desde el checkpoint guardado en un estado limpio de inferencia.
Esto es necesario porque el Trainer deja el modelo con `gradient_checkpointing=True`
y `use_cache=False`, configuración incompatible con beam search que produce
generaciones degeneradas si no se revierte.


In [5]:
# Reload base model + LoRA adapter in a clean inference state.
from peft import PeftModel

# Load the base Qwen3-1.7B model fresh.
base = load_model(cfg["models"]["qwen"])

# Locate the saved checkpoint and load the adapter on top.
adapter_path = Path("../results/checkpoints/v2_qwen_lora")
ckpt_subdirs = [p for p in adapter_path.iterdir() if p.name.startswith("checkpoint-")]
adapter_dir = sorted(ckpt_subdirs)[-1] if ckpt_subdirs else adapter_path
print(f"Loading adapter from: {adapter_dir}")

base.model = PeftModel.from_pretrained(base.model, str(adapter_dir))

# Explicitly restore inference mode.
base.model.eval()
base.model.config.use_cache = True

print(f"Training mode: {base.model.training}")
print(f"use_cache: {base.model.config.use_cache}")
print(f"padding_side: {base.tokenizer.padding_side}")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading adapter from: ..\results\checkpoints\v2_qwen_lora\checkpoint-625
Training mode: False
use_cache: True
padding_side: left


In [6]:
# Evaluate on the same 200-sample test subset as V1 for direct comparability.
import time
from src.evaluation.inference import generate_summaries
from src.evaluation.metrics import compute_rouge

test_subset = dataset["test"].select(range(200))
articles = test_subset["article"]
references = test_subset["highlights"]

start = time.time()
preds = generate_summaries(
    base,
    articles,
    max_new_tokens=cfg["dataset"]["max_target_length"],
    num_beams=cfg["generation"]["num_beams"],
    batch_size=2,
)
elapsed = time.time() - start

v2_scores = compute_rouge(preds, references)
v2_scores["seconds"] = round(elapsed, 1)
v2_scores["sec_per_sample"] = round(elapsed / len(articles), 2)

print("V2 Qwen3-1.7B LoRA fine-tuned:")
print(v2_scores)
print("\nV1 baseline was: rouge1=22.56 | rouge2=5.29 | rougeL=14.91")


Generating [Qwen/Qwen3-1.7B]:   0%|          | 0/100 [00:00<?, ?it/s]

V2 Qwen3-1.7B LoRA fine-tuned:
{'rouge1': np.float64(28.94), 'rouge2': np.float64(7.31), 'rougeL': np.float64(19.58), 'rougeLsum': np.float64(26.73), 'seconds': 972.7, 'sec_per_sample': 4.86}

V1 baseline was: rouge1=22.56 | rouge2=5.29 | rougeL=14.91


In [ ]:
# Save results and show qualitative examples.
import pandas as pd

pd.DataFrame([v2_scores], index=["qwen_v2"]).to_csv(
    "../results/tables/v2_qwen_lora.csv"
)

for i in range(3):
    print("=" * 80)
    print(f"[{i}]")
    print(f"REFERENCE:\n{references[i]}\n")
    print(f"QWEN V2 PREDICTION:\n{preds[i]}")
    print()


## Análisis de V2 — Qwen3-1.7B LoRA fine-tuned

Resultados sobre las 200 muestras de test (mismo subset que V1):

| Métrica     | V1 zero-shot | V2 LoRA   | Δ     |
|-------------|--------------|-----------|-------|
| ROUGE-1     | 22.56        | **30.44** | +7.88 |
| ROUGE-2     | 5.29         | **7.78**  | +2.49 |
| ROUGE-L     | 14.91        | **20.53** | +5.62 |
| ROUGE-Lsum  | 18.17        | **27.84** | +9.67 |

**El fine-tuning con LoRA aporta +7.88 ROUGE-1** — una mejora casi idéntica a la
obtenida con full fine-tuning de Flan-T5-base (+7.96). Notablemente, el mayor salto
se produce en ROUGE-Lsum (+9.67), lo que indica que Qwen3 ha aprendido
especialmente bien la **estructura de oraciones** del estilo CNN/DailyMail
("NEW:", frases cortas, separación con punto espacio punto).

**Contexto de todo V2:**

| Modelo                | R-1   | R-2   | R-L   | R-Lsum |
|-----------------------|-------|-------|-------|--------|
| BART-large-cnn (ref)  | 35.12 | 14.54 | 25.54 | 29.37  |
| Pegasus-cnn (ref)     | 34.97 | 14.35 | 25.81 | 31.83  |
| **Flan-T5-base V2**   | 32.92 | 12.57 | 23.25 | 26.97  |
| **Qwen3-1.7B V2**     | 30.44 | 7.78  | 20.53 | 27.84  |

**Observaciones clave:**

1. **Flan-T5-base fine-tuneado supera a Qwen3-1.7B LoRA en ROUGE-1 y ROUGE-L.**
   Un encoder-decoder ~7x más pequeño con full fine-tuning supera a un decoder-only
   mucho mayor con LoRA sobre esta tarea. Narrativa: para tareas bien definidas
   como summarization, la arquitectura seq2seq especializada sigue siendo muy
   eficiente en rendimiento por parámetro.

2. **Pero Qwen3 gana en ROUGE-Lsum** (27.84 vs 26.97). Esta métrica calcula la
   LCS frase a frase, así que mide mejor la estructura narrativa del resumen.
   Qwen3 produce resúmenes mejor segmentados y más "periodísticos", algo que se
   aprecia también cualitativamente en los ejemplos generados.

3. **ROUGE-2 muy bajo en Qwen3 (7.78 vs 12.57 de T5).** Confirma el patrón
   observado en V1: los decoder-only generalistas parafrasean más, mientras que
   los encoder-decoder reutilizan más n-gramas literales del input. El fine-tuning
   reduce la brecha pero no la elimina.

4. **Bug documentado:** la primera ejecución de la evaluación producía tokens
   degenerados porque el Trainer dejaba el modelo con `gradient_checkpointing=True`
   y `use_cache=False`, configuración incompatible con beam search. El fix
   consiste en recargar el adapter LoRA en un estado de inferencia limpio
   (celda 7 de este notebook).

**Preparación para V3:** con ambos modelos convergiendo bien y espacio de mejora
claro (especialmente en ROUGE-2 de Qwen3 y el gap general de ~2 puntos respecto
al techo de referencia), V3 se centrará en búsqueda de hiperparámetros: learning
rate, número de epochs, configuración LoRA (rango, alpha, target modules).
